In [1]:
import pandas as pd
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report

In [3]:
# Load the dataset
data = pd.read_csv("/content/AjwaOrMejdool.csv", delimiter=';')

In [4]:
# Inspect the data
print(data.head())

   Date Length (cm)  Date Diameter (cm)  Date Weight (g)  Pit Length (cm)  \
0               3.2                 2.0               12              2.2   
1               3.5                 1.8               11              1.9   
2               3.0                 1.7                9              2.0   
3               3.1                 2.0               10              1.9   
4               2.8                 1.8                9              1.9   

   Calories (Kcal)  Color Class (Ajwa or Medjool)  
0            41.28  Black                    Ajwa  
1            37.84  Black                    Ajwa  
2            30.96  Black                    Ajwa  
3            34.40  Black                    Ajwa  
4            30.96  Black                    Ajwa  


In [5]:

# Encode categorical features
color_encoder = LabelEncoder()
class_encoder = LabelEncoder()

In [6]:
data['Color'] = color_encoder.fit_transform(data['Color'])
data['Class (Ajwa or Medjool)'] = class_encoder.fit_transform(data['Class (Ajwa or Medjool)'])


In [7]:
# Verify the data types to ensure all columns are numeric
print(data.dtypes)

Date Length (cm)           float64
Date Diameter (cm)         float64
Date Weight (g)              int64
Pit Length (cm)            float64
Calories (Kcal)            float64
Color                        int64
Class (Ajwa or Medjool)      int64
dtype: object


In [8]:
# Define features and target variable
X = data[['Date Length (cm)', 'Date Diameter (cm)', 'Date Weight (g)', 'Pit Length (cm)', 'Calories (Kcal)', 'Color']]
y = data['Class (Ajwa or Medjool)']


In [9]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# Standardize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


# Define the DNN model
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')  # Use 'softmax' for multi-class classification
])


/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [10]:
# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])


In [11]:
# Train the model
model.fit(X_train, y_train, epochs=20, batch_size=16, validation_split=0.2)


Epoch 1/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.5833 - loss: 0.7069 - val_accuracy: 0.7500 - val_loss: 0.6566
Epoch 2/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step - accuracy: 0.5000 - loss: 0.6815 - val_accuracy: 0.7500 - val_loss: 0.6382
Epoch 3/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 111ms/step - accuracy: 0.8333 - loss: 0.6262 - val_accuracy: 0.7500 - val_loss: 0.6205
Epoch 4/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 136ms/step - accuracy: 0.5833 - loss: 0.6401 - val_accuracy: 1.0000 - val_loss: 0.6035
Epoch 5/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 141ms/step - accuracy: 0.6667 - loss: 0.6503 - val_accuracy: 1.0000 - val_loss: 0.5867
Epoch 6/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 135ms/step - accuracy: 0.8333 - loss: 0.5901 - val_accuracy: 1.0000 - val_loss: 0.5702
Epoch 7/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 134ms/step - accuracy: 0.7500 - loss: 0.5902 - val_accuracy: 1.0000 - val_loss: 0.5542
Epoch 8/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 168ms/step - accuracy: 0.8333 - loss: 0.5753 - val_accuracy: 1.0000 - val_loss: 0.5

In [12]:
# Evaluate the model
test_loss, test_acc = model.evaluate(X_test, y_test)
print("Test Accuracy:", test_acc * 100)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 1.0000 - loss: 0.4398
Test Accuracy: 100.0


In [14]:
# Predict and display classification report
y_pred = (model.predict(X_test) > 0.5).astype("int32")
print("Classification Report:\n", classification_report(y_test, y_pred))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 122ms/step
Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00         2
           1       1.00      1.00      1.00         2

    accuracy                           1.00         4
   macro avg       1.00      1.00      1.00         4
weighted avg       1.00      1.00      1.00         4



In [15]:
# Save the model in HDF5 format
model.save('dnn_date_classifier.h5')

In [16]:
# Convert to TFLite format
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

Saved artifact at '/tmp/tmp374go0rf'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 6), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  133836865659376: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133836865660432: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133839673735328: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133839394361360: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133836866214320: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133836866218368: TensorSpec(shape=(), dtype=tf.resource, name=None)


In [17]:
# Save the TFLite model
with open('dnn_date_classifier.tflite', 'wb') as f:
    f.write(tflite_model)


In [18]:

# Example prediction on new data
new_data = [[3.5, 1.8, 11, 1.9, 37.84, color_encoder.transform(['Black'])[0]]]  # Example data point
new_data = scaler.transform(new_data)
prediction = (model.predict(new_data) > 0.5).astype("int32")
predicted_class = class_encoder.inverse_transform(prediction)
print("Predicted Class:", predicted_class[0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
Predicted Class: Ajwa


/usr/local/lib/python3.10/dist-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/preprocessing/_label.py:153: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
